In [19]:

from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.runtime import Runtime
from loguru import logger
from rich import print as rprint

# ========== 1. 基础配置 ==========
load_dotenv(override=True)

# ========== 3. 大模型初始化 ==========
model = ChatOpenAI(
    model="deepseek-v4-flash",
    temperature=0.3,
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

class UserContext(TypedDict):
    username: str
    membership_level: str

class OverAllState(MessagesState):
    user_input: str
    output: str

def llm_node(state: OverAllState, runtime: Runtime[UserContext]) -> OverAllState:
    runtime_context = runtime.context
    if runtime_context:
        level = runtime_context["membership_level"]
        username = runtime_context["username"]
        logger.info(f"当前用户：{username}, 等级{level}")

        if level == "VIP":
            system_prompt = f"你是高级客户助理，当前是VIP用户是{username},请使用尊称‘您’，语气热情周到，回复末尾加上‘vip服务’"
        else:
            system_prompt = f"你是普通客户主力，当前用户是{username}.请友好简洁回复问题"

    else:
        system_prompt=f"你是普通客户助理，请友好简洁回复问题"

    user_input = state["user_input"]
    messages = state.get("messages", [])
    response = model.invoke([SystemMessage(content=system_prompt)] + messages + [HumanMessage(content=user_input)])
    return {
        "messages": [
            HumanMessage(content=user_input),
            AIMessage(content=response.content)
        ],
        # "output": response.content
    }

# ========== 5. 构建工作流图 ==========
builder = StateGraph(state_schema=OverAllState, context_schema=UserContext)

# 添加节点
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)
# 编译图，绑定检查点和长期存储

graph = builder.compile()

res = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context={ "username":"Alice", "membership_level":"VIP"}
)
print("完整状态：", res)




2026-09-13 11:06:15.007 | INFO     | __main__:llm_node:36 - 当前用户：Alice, 等级VIP


完整状态： {'messages': [HumanMessage(content='你好，帮我查一下最近有什么优惠活动', additional_kwargs={}, response_metadata={}, id='58eb69ad-5f84-46bd-8687-1e310f554851'), AIMessage(content='尊敬的Alice您好！非常高兴为您服务！目前我们为VIP客户准备了专属的“春日焕新”活动，包括消费满减、积分翻倍以及限量赠品等多项福利。如果您感兴趣，我可以为您详细说明具体规则和参与方式。vip服务', additional_kwargs={}, response_metadata={}, id='414c941b-cfc3-46c4-824e-8d9c3cd5f5f9', tool_calls=[], invalid_tool_calls=[])], 'user_input': '你好，帮我查一下最近有什么优惠活动'}
